In [1]:
!pip install -q transformers datasets accelerate sentencepiece

In [2]:
import pandas as pd
import numpy as np
import torch
import random

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected!")

GPU available: True
GPU: Tesla T4


In [5]:
from pathlib import Path
from google.colab import drive

# Mount Google Drive so Colab can access the CSV files.
drive.mount('/content/drive')

# Change this folder only if your CSV files are stored somewhere else.
DATA_DIR = Path('/content/drive/MyDrive/DS 340W')
train_path = DATA_DIR / 'train.csv'
validation_path = DATA_DIR / 'validation.csv'
test_path = DATA_DIR / 'test.csv'

# Check that all three files exist before attempting training.
for path in (train_path, validation_path, test_path):
    if not path.is_file():
        raise FileNotFoundError(f'File not found: {path}. Check DATA_DIR and filename.')

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(validation_path)
test_df = pd.read_csv(test_path)

print('Training shape:', train_df.shape)
print('Validation shape:', val_df.shape)
print('Test shape:', test_df.shape)
for name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    print(name, df['label'].value_counts().to_dict())


Mounted at /content/drive
Training shape: (4494, 3)
Validation shape: (642, 3)
Test shape: (1284, 3)
train {'real': 2352, 'fake': 2142}
validation {'real': 336, 'fake': 306}
test {'real': 672, 'fake': 612}


In [6]:
# Apply identical label normalization to all three splits.
label_map = {"fake": 0, "real": 1}
for split_name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    # Fail loudly if there are missing or unexpected labels; do not silently drop test rows.
    if df[["tweet", "label"]].isna().any().any():
        raise ValueError(f"Missing tweet or label in {split_name} data")
    df["tweet"] = df["tweet"].astype(str)
    df["label"] = df["label"].astype(str).str.lower().str.strip()
    df["labels"] = df["label"].map(label_map)
    if df["labels"].isna().any():
        raise ValueError(f"Unknown label in {split_name} data")
    df["labels"] = df["labels"].astype(int)
    print(split_name, df["labels"].value_counts().to_dict())

train {1: 2352, 0: 2142}
validation {1: 336, 0: 306}
test {1: 672, 0: 612}


In [7]:
# Make three distinct Hugging Face datasets.
train_dataset = Dataset.from_pandas(train_df[["tweet", "labels"]], preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[["tweet", "labels"]], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[["tweet", "labels"]], preserve_index=False)
print(train_dataset, val_dataset, test_dataset)

Dataset({
    features: ['tweet', 'labels'],
    num_rows: 4494
}) Dataset({
    features: ['tweet', 'labels'],
    num_rows: 642
}) Dataset({
    features: ['tweet', 'labels'],
    num_rows: 1284
})


In [8]:
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded!")

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

Tokenizer loaded!


In [9]:
# Use the same tokenizer settings on all three splits.
def tokenize_function(examples):
    return tokenizer(examples["tweet"], truncation=True, padding="max_length", max_length=128)

train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)
print("Tokenization complete")

Map:   0%|          | 0/4494 [00:00<?, ? examples/s]

Map:   0%|          | 0/642 [00:00<?, ? examples/s]

Map:   0%|          | 0/1284 [00:00<?, ? examples/s]

Tokenization complete


In [10]:
id2label = {
    0: "fake",
    1: "real"
}

label2id = {
    "fake": 0,
    "real": 1
}

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    torch_dtype=torch.float32
)

print("DeBERTa loaded!")
print("Model dtype:", next(model.parameters()).dtype)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  371MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den

DeBERTa loaded!
Model dtype: torch.float32


In [12]:
# Report macro metrics for fair model comparison and weighted metrics for continuity.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0)
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_p,
        "weighted_recall": weighted_r,
        "weighted_f1": weighted_f1,
    }

In [13]:
training_args = TrainingArguments(
    output_dir="./deberta_results_validation_corrected",

    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=50,

    save_total_limit=2,

    # IMPORTANT: keep both OFF
    fp16=False,
    bf16=False,

    report_to="none",

    seed=SEED
)

In [15]:
# Select the best checkpoint using validation data, never test data.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
)
print("Trainer ready; validation used for checkpoint selection")

Trainer ready; validation used for checkpoint selection


In [16]:
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Training examples:", len(train_tokenized))
print("Validation examples:", len(val_tokenized))
print("Test examples:", len(test_tokenized))

GPU available: True
GPU: Tesla T4
Training examples: 4494
Validation examples: 642
Test examples: 1284


In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
1,0.116415,0.104386,0.979751,0.979644,0.979779,0.979709,0.979758,0.979751,0.979752
2,0.072131,0.136995,0.968847,0.970984,0.967612,0.968687,0.969950,0.968847,0.968791
3,0.019912,0.127143,0.971963,0.973012,0.971172,0.971853,0.972382,0.971963,0.971935


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1686, training_loss=0.12291914143313434, metrics={'train_runtime': 594.709, 'train_samples_per_second': 22.67, 'train_steps_per_second': 2.835, 'total_flos': 886831716105216.0, 'train_loss': 0.12291914143313434, 'epoch': 3.0})

In [18]:
# Run this only after trainer.train() has completed.
# The test set is evaluated once, after selecting the best checkpoint on validation.
results = trainer.evaluate(eval_dataset=test_tokenized, metric_key_prefix="test")
print("\nDEBERTA FINAL TEST RESULTS")
print("=" * 40)
for key in ["test_accuracy", "test_macro_precision", "test_macro_recall", "test_macro_f1", "test_weighted_f1"]:
    print(f"{key}: {results[key] * 100:.2f}%")

Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
0.019912,0.135759,3,0.969626,0.969474,0.969669,0.969566,0.969643,0.969626,0.969629



DEBERTA FINAL TEST RESULTS
test_accuracy: 96.96%
test_macro_precision: 96.95%
test_macro_recall: 96.97%
test_macro_f1: 96.96%
test_weighted_f1: 96.96%


In [19]:
predictions_output = trainer.predict(test_tokenized)

predictions = np.argmax(
    predictions_output.predictions,
    axis=-1
)

true_labels = predictions_output.label_ids

print("DEBERTA CLASSIFICATION REPORT")
print("=" * 50)

print(
    classification_report(
        true_labels,
        predictions,
        target_names=["fake", "real"],
        digits=4
    )
)

DEBERTA CLASSIFICATION REPORT
              precision    recall  f1-score   support

        fake     0.9659    0.9706    0.9682       612
        real     0.9731    0.9688    0.9709       672

    accuracy                         0.9696      1284
   macro avg     0.9695    0.9697    0.9696      1284
weighted avg     0.9696    0.9696    0.9696      1284



In [20]:
cm = confusion_matrix(true_labels, predictions)

print("DEBERTA CONFUSION MATRIX")
print("=" * 40)

print(cm)

print("\nInterpretation:")
print(f"Fake correctly classified: {cm[0][0]}")
print(f"Fake incorrectly classified as real: {cm[0][1]}")
print(f"Real incorrectly classified as fake: {cm[1][0]}")
print(f"Real correctly classified: {cm[1][1]}")

DEBERTA CONFUSION MATRIX
[[594  18]
 [ 21 651]]

Interpretation:
Fake correctly classified: 594
Fake incorrectly classified as real: 18
Real incorrectly classified as fake: 21
Real correctly classified: 651


In [21]:
total_errors = np.sum(predictions != true_labels)
correct_predictions = len(true_labels) - total_errors

print("Total test examples:", len(true_labels))
print("Correct predictions:", correct_predictions)
print("Incorrect predictions:", total_errors)
print(f"Error rate: {(total_errors / len(true_labels)) * 100:.2f}%")

Total test examples: 1284
Correct predictions: 1245
Incorrect predictions: 39
Error rate: 3.04%


In [22]:
error_indices = np.where(predictions != true_labels)[0]

errors_df = test_df.iloc[error_indices].copy()

errors_df["true_label"] = [
    id2label[x] for x in true_labels[error_indices]
]

errors_df["predicted_label"] = [
    id2label[x] for x in predictions[error_indices]
]

print("Number of misclassified tweets:", len(errors_df))

errors_df[
    ["tweet", "true_label", "predicted_label"]
].head(20)

Number of misclassified tweets: 39


,tweet,true_label,predicted_label
17,Testing doesn’t create the virus. Even without...,fake,real
35,A person who has been released from quarantine...,real,fake
121,Important thread on the risk of dietary supple...,fake,real
146,Another reason is the casual attitude of peopl...,real,fake
187,U.S. sees 35K new COVID cases; national total ...,fake,real
198,The information circulating on social media th...,real,fake
204,#eVIN is presently being implemented by the Ce...,real,fake
248,"In AURANGABA, Covid antibody testing in Privat...",fake,real
250,There's now 3 different #coronavirus vaccines ...,fake,real
253,ICMR is honoured to have carried out testing o...,real,fake


In [23]:
comparison = pd.DataFrame({
    "Model": [
        "RoBERTa",
        "DistilBERT",
        "Stacking Ensemble",
        "CNN-LSTM",
        "DeBERTa"
    ],

    "Test Accuracy (%)": [
        97.51,
        96.65,
        93.85,
        91.51,
        results["test_accuracy"] * 100
    ]
})

comparison = comparison.sort_values(
    "Test Accuracy (%)",
    ascending=False
).reset_index(drop=True)

comparison

,Model,Test Accuracy (%)
0,RoBERTa,97.510000
1,DeBERTa,96.962617
2,DistilBERT,96.650000
3,Stacking Ensemble,93.850000
4,CNN-LSTM,91.510000
